# 🎛️ Phase 2: Finding the Best Model Settings (Hyperparameter Tuning)
**Author:** Kyrylo Kudrevych

### 🎯 What are we doing here?
Machine learning models come with default settings, but they are rarely perfect for our specific dataset. Think of it like tuning a guitar before playing a concert.

In this notebook, we are using a library called **Optuna**. It acts like a smart robot that tests hundreds of different combinations of settings (hyperparameters) for four different algorithms to find the exact combination that gives us the lowest Mean Absolute Error (MAE).

### Best Practices Applied Here:
* **Log Transformation (`np.log1p`):** Rent prices are heavily skewed (a few luxury apartments pull the average up). We apply a logarithmic transformation to our prices to compress them. This makes it much easier for our models to find patterns.
* **Preventing Data Leakage:** We strictly split our data into a Training set (80%) and a Hidden Test set (20%). Notice how we calculate the median floor number *only* from the training data, and then apply it to the test data. This ensures our model never "peeks" at the future!

In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load the clean data
df = pd.read_parquet('notebooks_data/data_ready_for_ml.parquet')

# 2. Separate features (X) and target (y)
X = df.drop(columns=['true_price'])

# 3. Apply safe Log Transformation to the target
y_log = np.log1p(df['true_price'])

# 4. Split the data safely
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=52)

# 5. Fill missing floors using ONLY the training data's median
train_floor_median = X_train['floor_num'].median()
X_train['floor_num'] = X_train['floor_num'].fillna(train_floor_median)
X_test['floor_num'] = X_test['floor_num'].fillna(train_floor_median)

In [13]:
import optuna
import warnings
from sklearn.metrics import mean_absolute_error

from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

## 🌲 1. Tuning Random Forest
Random Forest builds hundreds of independent decision trees and averages their guesses. It is very stable, but can be quite slow and heavy. Let's see what settings work best.

*(Note: We use `np.expm1` to reverse our log transformation so we can calculate the real error in PLN).*

In [14]:
import optuna
import warnings
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

def rf_objective(trial):
    rf_param = {
        'random_state': 52,
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', [1.0, 'sqrt', 'log2'])
    }

    rf_model = RandomForestRegressor(**rf_param)
    rf_model.fit(X_train, y_train)

    rf_y_pred = rf_model.predict(X_test)

    # Reverse the log1p transformation
    rf_final_predictions = np.expm1(rf_y_pred)
    rf_final_y_test = np.expm1(y_test)

    return mean_absolute_error(rf_final_y_test, rf_final_predictions)

def start_rf_study():
    rf_study = optuna.create_study(direction='minimize', study_name="RandomForest_Housing")
    print("Starting Random Forest Optuna search... (This might take a minute)")
    rf_study.optimize(rf_objective, n_trials=200)

    print("\n--- RANDOM FOREST OPTIMIZATION FINISHED ---")
    print(f"Best MAE: {rf_study.best_value:.2f} PLN")
    print("Best Parameters:", rf_study.best_params)

# start_rf_study() # Uncomment to run

# --- SAVED RESULTS ---
# Best MAE: 329.51 PLN
# Best Parameters: {'n_estimators': 300, 'max_depth': 19, 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_features': 1.0}

## ⚡ 2. Tuning LightGBM
LightGBM is incredibly fast and learns by building trees sequentially (each tree tries to fix the mistakes of the previous one). It is highly sensitive to hyperparameters.

In [16]:
import lightgbm as lgb

def lgb_objective(trial):
    lgb_param = {
        'boosting_type': 'gbdt',
        'random_state': 52,
        'verbose': -1,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 500, 3000),
        'num_leaves': trial.suggest_int('num_leaves', 5, 25),
        'max_depth': trial.suggest_int('max_depth', 2, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 15, 60),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
    }

    lgb_model = lgb.LGBMRegressor(**lgb_param)

    lgb_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        eval_metric='mae',
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    lgb_y_pred = lgb_model.predict(X_test)

    # Reverse the log1p transformation
    lgb_final_predictions = np.expm1(lgb_y_pred)
    lgb_final_y_test = np.expm1(y_test)

    return mean_absolute_error(lgb_final_y_test, lgb_final_predictions)

def start_lgb_study():
    lgb_study = optuna.create_study(direction='minimize', study_name="LightGBM_Housing")
    print("Starting LightGBM Optuna search...")
    lgb_study.optimize(lgb_objective, n_trials=200)

    print("\n--- LIGHTGBM OPTIMIZATION FINISHED ---")
    print(f"Best MAE: {lgb_study.best_value:.2f} PLN")
    print("Best Parameters:", lgb_study.best_params)

# start_lgb_study() # Uncomment to run

# --- SAVED RESULTS ---
# Best MAE: 311.56 PLN
# Best Parameters: {'learning_rate': 0.077, 'n_estimators': 1048, 'num_leaves': 6, 'max_depth': 6, 'min_child_samples': 42, 'subsample': 0.64, 'colsample_bytree': 0.56}

## 🚀 3. Tuning XGBoost
XGBoost is the classic heavyweight champion of machine learning competitions. It takes a bit longer to train than LightGBM but often finds very deep patterns.

In [17]:
import xgboost as xgb

def xgb_objective(trial):
    xgb_param = {
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'random_state': 52,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 500, 2500),
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 40),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'verbosity': 0
    }

    xgb_model = xgb.XGBRegressor(**xgb_param)

    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    xgb_y_pred = xgb_model.predict(X_test)

    # Reverse the log1p transformation
    xgb_final_predictions = np.expm1(xgb_y_pred)
    xgb_final_y_test = np.expm1(y_test)

    return mean_absolute_error(xgb_final_y_test, xgb_final_predictions)

def start_xgb_study():
    xgb_study = optuna.create_study(direction='minimize', study_name="XGBoost_Housing")
    print("Starting XGBoost Optuna search...")
    xgb_study.optimize(xgb_objective, n_trials=200)

    print(f"\n--- XGBOOST OPTIMIZATION FINISHED ---")
    print(f"Best MAE: {xgb_study.best_value:.2f} PLN")
    print("Best Parameters:", xgb_study.best_params)

# start_xgb_study() # Uncomment to run

# --- SAVED RESULTS ---
# Best MAE: 312.71 PLN
# Best Parameters: {'learning_rate': 0.013, 'n_estimators': 1090, 'max_depth': 4, 'min_child_weight': 24, 'subsample': 0.50, 'colsample_bytree': 0.61}

## 🐱 4. Tuning CatBoost
CatBoost is famous for being incredibly accurate out-of-the-box, especially with categorical data. Let's see if tuning its parameters pushes it to first place.

In [18]:
from catboost import CatBoostRegressor

def catboost_objective(trial):
    cat_param = {
        'loss_function': 'MAE',
        'eval_metric': 'MAE',
        'random_state': 52,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'iterations': trial.suggest_int('iterations', 500, 2500),
        'depth': trial.suggest_int('depth', 3, 7),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'verbose': False
    }

    cat_model = CatBoostRegressor(**cat_param)

    cat_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        early_stopping_rounds=50,
        verbose=False
    )

    cat_y_pred = cat_model.predict(X_test)

    # Reverse the log1p transformation
    cat_final_predictions = np.expm1(cat_y_pred)
    cat_final_y_test = np.expm1(y_test)

    return mean_absolute_error(cat_final_y_test, cat_final_predictions)

def start_cat_study():
    cat_study = optuna.create_study(direction='minimize', study_name="CatBoost_Housing")
    print("Starting CatBoost Optuna search...")
    cat_study.optimize(catboost_objective, n_trials=200)

    print(f"\n--- CATBOOST OPTIMIZATION FINISHED ---")
    print(f"Best MAE: {cat_study.best_value:.2f} PLN")
    print("Best Parameters:", cat_study.best_params)

# start_cat_study() # Uncomment to run

# --- SAVED RESULTS ---
# Best MAE: 309.12 PLN
# Best Parameters: {'learning_rate': 0.057, 'iterations': 1315, 'depth': 7, 'l2_leaf_reg': 6.01, 'subsample': 0.85}